## 0. Setup shared by the three validation notebooks
Paths are relative to the repository root. The statistics come from `tools/validation_metrics.py`
(Ministry of Transport guideline, draft 6, September 2024); results go to `Output/validation_mot/`.

In [1]:
import os, sys, warnings
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'
sys.path.insert(0, 'tools')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import validation_metrics as vm
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6'
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/validation_mot'; FIG = 'Output/figures'
# Time window (wave 1b): AM 06:00-09:00 is the base and the default; PM 16:00-19:00 and midday 10:00-14:00 with the environment variable NOFIT_PERIOD
PERIOD = os.environ.get('NOFIT_PERIOD', 'AM')
HOURS, SUF = {'AM': ([6, 7, 8], ''), 'PM': ([16, 17, 18], '_pm'), 'MD': ([10, 11, 12, 13], '_md')}[PERIOD]
PLABEL = {'AM': 'AM 06:00-09:00', 'PM': 'PM 16:00-19:00', 'MD': 'midday 10:00-14:00'}[PERIOD]
W = {'AM': '06-09', 'PM': '16-19', 'MD': '10-14'}[PERIOD]
PSUF = '' if PERIOD == 'AM' else '_' + PERIOD            # suffix of the validation output files of the other periods
def vfile(name): return f'{OUT}/{name}{PSUF}.csv'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
def need(*paths):
    for p in paths: assert not is_pointer(p), f'{p} is a Git LFS pointer: python3 tools/lfs_pull.py {p}'

SUMMARY = []      # one row per guideline check; written to Output/validation_mot/summary_<stage>.csv
def add_summary(test, ref, item, level, criterion, result, status, explanation, data):
    SUMMARY.append({'period': PERIOD, 'test': test, 'guideline': ref, 'item': item, 'level': level, 'criterion': criterion,
                    'result': result, 'status': status, 'explanation': explanation, 'data (vintage)': data})

def scatter_origin(ax, obs, mod, title, xlabel, ylabel, logscale=False, color=BLUE):
    """Guideline format: observed on x, model on y, regression through the origin with equation and R2."""
    o = np.asarray(obs, float); m = np.asarray(mod, float); ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    ax.scatter(o, m, s=10, color=color, alpha=0.55, linewidths=0)
    hi = max(o.max(), m.max()) * 1.05 if len(o) else 1
    ax.plot([0, hi], [0, hi], color=MUTED, lw=0.8, ls=':')
    b = vm.slope_origin(o, m); r2 = vm.r2_correl(o, m)
    ax.plot([0, hi], [0, b * hi], color=ORANGE, lw=1.5)
    ax.text(0.04, 0.95, f'y = {b:.2f}x   R² = {r2:.2f}   n = {len(o)}', transform=ax.transAxes, va='top', fontsize=9, color=INK)
    if logscale: ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    style_ax(ax, title, xlabel, ylabel)

def save_fig(fig, name):
    fig.savefig(f'{FIG}/{name.replace(".png", PSUF + ".png")}', dpi=150, bbox_inches='tight', facecolor='white'); plt.close(fig)

## 0. Zones, keys and settlement / sector classes (shared by this notebook's tests)

* **Super-zone** of a TAZ: the keys table the chain uses (`Input/Matrices/1270_..._keys.csv`, decision of 4 October 2026).
* **Locality type** (guideline 3.2: urban / suburban / rural): `SETTL_SIZE` of `Zonal_2020.csv`, 3 = urban, 2 = suburban, 1 = rural. This reading is a working assumption from the densities of the three classes
  (population per area unit 2,715 / 319 / 54); the field has no dictionary in the repository.
* **Sector of a household / person** (guideline 3.2): `HHTYPE` of the household table: Arab = `arab`, Haredi = `orto`, Jewish non-Haredi = `secular` + `religious` + `other` (decision D2). The matrices carry no sector, so where a matrix is split by sector
  the TAZ is assigned the sector with the largest share of its population (`SECTOR_ARA`, `SECTOR_ORT`, `SECTOR_REST` of the zonal file): a stated proxy.

In [2]:
def afile(name): return f'{OUT}/{name}.csv'                                   # period-independent outputs (T1, T2)
KEYS = 'Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv'
need(KEYS, 'Input/TAZ_2636_Keys.xlsx', 'Input/Submatrix_tazs.xlsx', 'Input/Corridor_TAZ_Agg_V2.xlsx', 'Input/TAZ_GSnew.csv')
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='cp1255').set_index('TAZ_ID'); zon25 = pd.read_csv('Input/Zonal_BU_2025.csv', encoding='cp1255').set_index('TAZ_ID')
TAZ = [int(c) for c in pd.read_csv('Output/ths2017/two_mode/car_taz.csv', index_col=0).columns]
keys_raw = pd.read_csv(KEYS, encoding='windows-1255')
taz_sz = keys_raw[['TAZ_NUMBER', 'SZ_NEW']].dropna().astype(int).drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')['SZ_NEW'].reindex(TAZ).astype(int)
kd = keys_raw[['TAZ_1270', 'TAZ_NUMBER']].dropna(subset=['TAZ_NUMBER']).astype({'TAZ_NUMBER': int}).drop_duplicates('TAZ_NUMBER')
taz_1250 = kd.set_index('TAZ_NUMBER')['TAZ_1270'].reindex(TAZ); Z1250 = sorted(taz_1250.unique())
d_ = pd.DataFrame({'z': taz_1250.values, 'g': taz_sz.values, 'pop': zon.loc[TAZ, 'POPULATION'].values}, index=TAZ)
sz_of_1250 = pd.Series({z: int(g.groupby('g')['pop'].sum().idxmax()) for z, g in d_.groupby('z')})
SZL = sorted(taz_sz.unique()); sz_names = pd.read_csv('Input/sz_localities.csv').set_index('SZ_NEW')['main localities']
LOCNAME = {1: 'rural', 2: 'suburban', 3: 'urban'}
taz_loc = zon.loc[TAZ, 'SETTL_SIZE'].map(LOCNAME)
SECTORS = ['Jewish non-Haredi', 'Haredi', 'Arab']
sec_share = zon.loc[TAZ, ['SECTOR_REST', 'SECTOR_ORT', 'SECTOR_ARA']].fillna(0); sec_share.columns = SECTORS
taz_sector = sec_share.idxmax(axis=1)                                                # proxy: dominant sector of the TAZ population
sub = pd.read_excel('Input/Submatrix_tazs.xlsx'); taz_area = sub.set_index('TAZ')['AggAreaCode']
corr_taz = set(sub.loc[sub['IsLRT_Corridor'] == 1, 'TAZ']) & set(TAZ)
v2 = pd.read_excel('Input/Corridor_TAZ_Agg_V2.xlsx', sheet_name='TazAgg').set_index('TAZ')['AggCode']; gs = pd.read_csv('Input/TAZ_GSnew.csv').set_index('TAZ')['GS']
SECTOR_OF_HHTYPE = {'arab': 'Arab', 'orto': 'Haredi'}                                # everything else: Jewish non-Haredi
print(f'{len(TAZ)} study TAZs, {len(SZL)} super-zones, {len(Z1250)} 1250-zones; locality types {taz_loc.value_counts().to_dict()}; dominant sector {taz_sector.value_counts().to_dict()}')

778 study TAZs, 36 super-zones, 396 1250-zones; locality types {'urban': 400, 'suburban': 219, 'rural': 159}; dominant sector {'Jewish non-Haredi': 568, 'Arab': 200, 'Haredi': 10}


# Ministry of Transport validation, wave 2a: zone system, population, trip rates, intra-zonal share (T1, T2, T3, T6)

**Question.** Are the definitions, the population control and the trip rates under the matrices consistent with each other and with the guideline's checks 3.1-3.5? These tests are mostly *consistency
checks*: the matrices are built from the survey, so a match with the survey says whether a step we added (zone allocation, growth to 2022) preserved it, not that a model reproduces data. The comparison with CBS
(employed persons, work location, car availability from the social survey) is not possible: CBS tables are not in the repository (standing data request 6).
Frozen version: 23 September 2026 rebuild; AM 06:00-09:00, PM peak 16:00-19:00 and midday 10:00-14:00 (wave 1b).

## 1. T1 - the zone system and the segment definitions (guideline 3.1, 3.2)

In [3]:
import geopandas as gpd
shp = gpd.read_file('Input/TAZ_North/TAZ_North.shp').drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')
rows = []
for s_ in SZL:
    t = [x for x in TAZ if taz_sz[x] == s_]; p = zon.loc[t, 'POPULATION']
    rows.append({'super-zone': s_, 'main localities': sz_names.get(s_, ''), 'TAZs': len(t), '1250-zones (dominant)': int((sz_of_1250 == s_).sum()), 'LRT-corridor TAZs': sum(x in corr_taz for x in t),
                 'population 2020': p.sum(), 'employment 2020': zon.loc[t, 'EMPL_TOT'].sum(), 'Arab share': (p * sec_share.loc[t, 'Arab']).sum() / p.sum() if p.sum() else np.nan,
                 'Haredi share': (p * sec_share.loc[t, 'Haredi']).sum() / p.sum() if p.sum() else np.nan,
                 **{f'{n} share of population': p[taz_loc.loc[t] == n].sum() / p.sum() if p.sum() else np.nan for n in ['urban', 'suburban', 'rural']}})
t1 = pd.DataFrame(rows); t1.to_csv(afile('T1_zone_hierarchy'), index=False, float_format='%.4f')
levels = pd.DataFrame([{'level': 'TAZ', 'zones': len(TAZ), 'covers': 'all 778 study TAZs'}, {'level': 'super-zone (keys table)', 'zones': len(SZL), 'covers': 'all TAZs'},
                       {'level': '1250-zone', 'zones': len(Z1250), 'covers': 'all TAZs (survey / cellular unit)'}, {'level': 'GS zone', 'zones': int(gs.reindex(TAZ).nunique()), 'covers': f'{int(gs.reindex(TAZ).notna().sum())} TAZs'},
                       {'level': 'sub-area (28)', 'zones': int(taz_area.reindex(TAZ).nunique()), 'covers': f'{int(taz_area.reindex(TAZ).notna().sum())} TAZs'},
                       {'level': 'V2 area (25)', 'zones': int(v2.reindex(TAZ).nunique()), 'covers': f'{int(v2.reindex(TAZ).notna().sum())} TAZs'}])
levels.to_csv(afile('T1_levels'), index=False)
def nesting(name, mp):
    m = pd.DataFrame({'g': mp.reindex(TAZ), 'sz': taz_sz}).dropna(); n = m.groupby('g')['sz'].nunique()
    return {'level': name, 'groups': int(n.size), 'groups spanning more than one super-zone': int((n > 1).sum()), 'TAZs in those groups': int(m['g'].isin(n[n > 1].index).sum())}
z_sz_zonal = zon.loc[TAZ, 'SZ_NEW'].astype(int)
nest = pd.DataFrame([nesting('sub-area (28)', taz_area), nesting('V2 area (25)', v2), nesting('GS zone', gs), nesting('1250-zone', pd.Series(taz_1250.values, index=TAZ)),
                     {'level': 'TAZ own super-zone vs dominant super-zone of its 1250-zone', 'groups': np.nan, 'groups spanning more than one super-zone': np.nan, 'TAZs in those groups': int((taz_sz.values != taz_1250.map(sz_of_1250).values).sum())},
                     {'level': 'keys table vs Zonal_2020.csv super-zone', 'groups': np.nan, 'groups spanning more than one super-zone': np.nan, 'TAZs in those groups': int((taz_sz.values != z_sz_zonal.values).sum())}])
nest.to_csv(afile('T1_nesting_checks'), index=False); print(levels.to_string(index=False)); print(nest.to_string(index=False))
sect = pd.DataFrame({'HHTYPE': ['arab', 'orto', 'religious', 'secular', 'other'], 'sector used': ['Arab', 'Haredi', 'Jewish non-Haredi', 'Jewish non-Haredi', 'Jewish non-Haredi']}); sect.to_csv(afile('T1_sector_coding'), index=False)

                  level  zones                            covers
                    TAZ    778                all 778 study TAZs
super-zone (keys table)     36                          all TAZs
              1250-zone    396 all TAZs (survey / cellular unit)
                GS zone     25                          778 TAZs
          sub-area (28)     28                          205 TAZs
           V2 area (25)     25                          174 TAZs
                                                     level  groups  groups spanning more than one super-zone  TAZs in those groups
                                             sub-area (28)    28.0                                      13.0                   136
                                              V2 area (25)    25.0                                      15.0                   124
                                                   GS zone    25.0                                      11.0                   537
                     

In [4]:
tz = shp.loc[TAZ].copy(); tz['SZ'] = taz_sz.values; tz['loc'] = taz_loc.values; tz['corr'] = [t in corr_taz for t in TAZ]
fig, axes = plt.subplots(1, 3, figsize=(16, 6.2), facecolor='white')
sz_c = tz.dissolve(by='SZ'); import matplotlib.cm as cm
tz.plot(ax=axes[0], column='SZ', cmap='tab20', linewidth=0.1, edgecolor='white'); sz_c.boundary.plot(ax=axes[0], color=INK, linewidth=0.5)
for s_, r in sz_c.iterrows(): axes[0].annotate(str(s_), r.geometry.representative_point().coords[0], fontsize=7, ha='center')
axes[0].set_title('36 super-zones (keys table)', fontsize=11)
tz.plot(ax=axes[1], column='loc', cmap='viridis', legend=True, linewidth=0.1, edgecolor='white'); axes[1].set_title('Locality type (SETTL_SIZE: 3 urban, 2 suburban, 1 rural)', fontsize=11)
tz.plot(ax=axes[2], color='#dddddd', linewidth=0.1, edgecolor='white'); tz[tz['corr']].plot(ax=axes[2], color=ORANGE, linewidth=0.1, edgecolor='white'); sz_c.boundary.plot(ax=axes[2], color=INK, linewidth=0.4); axes[2].set_title('LRT-corridor TAZs (orange, 119 in the sub-area file)', fontsize=11)
for ax in axes: ax.set_axis_off()
fig.tight_layout(); save_fig(fig, 'mot_T1_zone_system.png')
add_summary('T1', '3.1', 'zone hierarchy 778 TAZ / 36 super-zones / 396 1250-zones / 28 sub-areas / 25 V2 areas documented, with a map', 'all', 'presented and approved by the steering committee',
            f'{len(TAZ)} TAZs, {len(SZL)} super-zones, {len(Z1250)} 1250-zones; the map and the nesting checks are in the workbook', 'documented', 'The guideline asks the steering committee to approve the geography; the core / ring / sector split of the Haifa metropolis is not defined in the repository.', 'keys table; Submatrix_tazs; Corridor_TAZ_Agg_V2')
for _, r in nest.iloc[:4].iterrows():
    add_summary('T1', '3.1', f'nesting: {r["level"]} inside super-zones', 'zone system', 'each group inside one super-zone', f'{int(r["groups spanning more than one super-zone"])} of {int(r["groups"])} groups span more than one super-zone ({int(r["TAZs in those groups"])} TAZs)',
                'pass' if r['groups spanning more than one super-zone'] == 0 else 'finding', 'A level that is not nested in the super-zones cannot be summed to them without a rule.' if r['groups spanning more than one super-zone'] else '', 'zone tables')
add_summary('T1', '3.2', 'sector coding (Jewish non-Haredi / Haredi / Arab) and locality type (urban / suburban / rural)', 'household / TAZ', 'segments defined', 'sector from HHTYPE (decision D2); locality type from SETTL_SIZE (assumed 3 urban, 2 suburban, 1 rural)', 'documented',
            'The reading of SETTL_SIZE is an assumption from the class densities; confirm with the owner of the zonal file.', 'THS household table; Zonal_2020.csv')

## 2. T2 - the population the survey expands to, against the zonal population (guideline 3.3)

**Test.** Residents by super-zone (±5 %), by sector (±5 %), households and average household size (±5 %), expanded with the survey weight (`new_wf` of the trips file, constant within a person and equal to the household weight of
`households_with_weights.csv`) and assigned to the TAZ polygon that contains the household's coordinates, against `Zonal_2020.csv` (observed 2020, from the national statistics). The survey is 2017/18 and the zonal base 2020; the gap is stated, and a second row removes it
with one overall scale factor, which judges the spatial *pattern* only. Car availability (`HHVEHICLE`) cannot be compared: the zonal file has no car field (CBS social survey requested).

In [5]:
TRIPS = 'Input/THS_2017-2018/trips_ths_2017.xlsx'; need(TRIPS, 'Input/THS_2017-2018/HHfinal.csv', 'Input/THS_2017-2018/PersonsFin2.csv', 'Input/Matrices/households_with_weights.csv')
tr = pd.read_excel(TRIPS); tr['INDIVID'] = tr['HHID3'] * 100 + tr['IndID3']
pw = tr.groupby('INDIVID').agg(new_wf=('new_wf', 'first'), nwf_n=('new_wf', 'nunique'))
per = pd.read_csv('Input/THS_2017-2018/PersonsFin2.csv', encoding='utf-8-sig'); hh = pd.read_csv('Input/THS_2017-2018/HHfinal.csv', encoding='utf-8-sig'); hw = pd.read_csv('Input/Matrices/households_with_weights.csv')
assert pw.nwf_n.max() == 1
P = pw.join(per.set_index('INDIVID')[['HHID']], how='left'); n_drop = int(P.HHID.isna().sum()); P = P.dropna(subset=['HHID']); P['HHID'] = P['HHID'].astype(int)
chk = P.merge(hw[['HHID', 'wf_new']], on='HHID', how='left'); print(f'persons in the trips file {len(pw):,}; without a person record {n_drop}; person weight vs household weight wf_new: max abs difference {np.abs(chk.new_wf - chk.wf_new).max():.2e}')
hh['sector'] = hh['HHTYPE'].map(SECTOR_OF_HHTYPE).fillna('Jewish non-Haredi')
pts = gpd.GeoDataFrame(hh, geometry=gpd.points_from_xy(hh.lon, hh.lat), crs=4326).to_crs(shp.crs)
home = gpd.sjoin(pts, gpd.GeoDataFrame(geometry=shp.geometry.values, index=shp.index, crs=shp.crs).reset_index().rename(columns={'index': 'TAZ_NUMBER'}), how='left', predicate='within').drop_duplicates('HHID').set_index('HHID')
hh = hh.set_index('HHID').join(home['TAZ_NUMBER'].rename('home_taz'))
P = P.join(hh[['home_taz', 'sector', 'HHSIZE', 'HHVEHICLE']], on='HHID'); P['home_taz'] = P['home_taz'].where(P['home_taz'].isin(TAZ)); P['sz'] = P['home_taz'].map(taz_sz); P['loc'] = P['home_taz'].map(taz_loc)
print(f'households located in a study TAZ: {hh.home_taz.isin(TAZ).mean():.1%} of {len(hh):,}; persons located: {P.home_taz.notna().mean():.1%}')
exp_pop = P.groupby('sz')['new_wf'].sum().reindex(SZL).fillna(0); zon_pop = zon.loc[TAZ, 'POPULATION'].groupby(taz_sz.values).sum().reindex(SZL)
t2 = pd.DataFrame({'zonal population 2020': zon_pop, 'THS-expanded residents': exp_pop}); t2['diff %'] = vm.pct_diff(t2.iloc[:, 0], t2.iloc[:, 1]); t2['within ±5 %'] = t2['diff %'].abs() <= 5
k_ = t2.iloc[:, 0].sum() / t2.iloc[:, 1].sum(); t2['THS scaled by one factor'] = t2['THS-expanded residents'] * k_; t2['diff % after one factor'] = vm.pct_diff(t2.iloc[:, 0], t2['THS scaled by one factor']); t2['within ±5 % after one factor'] = t2['diff % after one factor'].abs() <= 5
t2.index.name = 'super-zone'; t2.to_csv(afile('T2_residents_by_superzone'), float_format='%.2f')
fs = vm.fit_stats(t2['zonal population 2020'], t2['THS-expanded residents']); fs_k = vm.fit_stats(t2['zonal population 2020'], t2['THS scaled by one factor'])
print(f'residents: zonal {fs["obs_total"]:,.0f} vs THS {fs["mod_total"]:,.0f} ({fs["total_diff_pct"]:+.1f} %); super-zone R2 {fs["R2"]:.3f}, slope {fs["slope_origin"]:.3f}, RMSE% {fs["RMSE_pct"]:.1f}; within ±5 %: {t2["within ±5 %"].sum()} of {len(t2)}; after one factor ({k_:.3f}): within ±5 %: {t2["within ±5 % after one factor"].sum()}, RMSE% {fs_k["RMSE_pct"]:.1f}')
tazp = P.groupby('home_taz')['new_wf'].sum().reindex(TAZ).fillna(0); fs_taz = vm.fit_stats(zon.loc[TAZ, 'POPULATION'], tazp)
print(f'TAZ level: R2 {fs_taz["R2"]:.3f}, slope {fs_taz["slope_origin"]:.3f}, RMSE% {fs_taz["RMSE_pct"]:.0f}')
# sector
zsec = pd.DataFrame({'Jewish non-Haredi': (zon.loc[TAZ, 'POPULATION'] * sec_share['Jewish non-Haredi']), 'Haredi': (zon.loc[TAZ, 'POPULATION'] * sec_share['Haredi']), 'Arab': (zon.loc[TAZ, 'POPULATION'] * sec_share['Arab'])})
rows = []
for sct in SECTORS:
    zs = zsec[sct].groupby(taz_sz.values).sum().reindex(SZL).fillna(0); es = P[P.sector == sct].groupby('sz')['new_wf'].sum().reindex(SZL).fillna(0)
    rows.append({'sector': sct, 'zonal residents': zs.sum(), 'THS residents': es.sum(), 'diff %': 100 * (es.sum() - zs.sum()) / zs.sum(), 'zonal share': zs.sum() / zsec.values.sum(), 'THS share': es.sum() / exp_pop.sum(),
                 'super-zones with > 1,000 zonal residents': int((zs > 1000).sum()), 'of them within ±5 % after one factor': int((np.abs(vm.pct_diff(zs[zs > 1000].values, (es[zs > 1000] * (zs.sum() / es.sum())).values)) <= 5).sum())})
t2s = pd.DataFrame(rows); t2s.to_csv(afile('T2_residents_by_sector'), index=False, float_format='%.4f'); print(t2s.round(3).to_string(index=False))
# households and household size
hhw = P.groupby('HHID').agg(w=('new_wf', 'first'), size=('HHSIZE', 'first'), sz=('sz', 'first'), npers=('new_wf', 'size')).dropna(subset=['sz'])
exp_hh = hhw.groupby('sz')['w'].sum().reindex(SZL).fillna(0); zon_hh = zon.loc[TAZ, 'HH_TOT'].groupby(taz_sz.values).sum().reindex(SZL)
exp_size = (hhw['w'] * hhw['size']).groupby(hhw['sz']).sum().reindex(SZL) / exp_hh; zon_size = zon_pop / zon_hh
t2h = pd.DataFrame({'zonal households': zon_hh, 'THS-expanded households': exp_hh, 'zonal household size': zon_size, 'THS household size': exp_size})
t2h['households diff %'] = vm.pct_diff(t2h['zonal households'], t2h['THS-expanded households']); t2h['size diff %'] = vm.pct_diff(t2h['zonal household size'], t2h['THS household size']); t2h['size within ±5 %'] = t2h['size diff %'].abs() <= 5
t2h.index.name = 'super-zone'; t2h.to_csv(afile('T2_households_by_superzone'), float_format='%.3f')
tot_size = (hhw['w'] * hhw['size']).sum() / hhw['w'].sum(); print(f'households: zonal {zon_hh.sum():,.0f}, THS {exp_hh.sum():,.0f}; average size zonal {zon_pop.sum() / zon_hh.sum():.2f} vs THS {tot_size:.2f}; super-zones with the size within ±5 %: {int(t2h["size within ±5 %"].sum())} of {len(t2h)}')
car_av = hhw.join(P.groupby('HHID')['HHVEHICLE'].first()); car_av['cars'] = car_av['HHVEHICLE'].clip(upper=2).map({0: '0 cars', 1: '1 car', 2: '2+ cars'})
cav = (car_av.groupby('cars')['w'].sum() / car_av['w'].sum()).rename('share of households'); cav.to_csv(afile('T2_car_availability_descriptive'), float_format='%.4f'); print(cav.round(3).to_dict())

persons in the trips file 16,401; without a person record 13; person weight vs household weight wf_new: max abs difference 0.00e+00
households located in a study TAZ: 99.6% of 5,098; persons located: 99.6%
residents: zonal 2,539,095 vs THS 2,670,719 (+5.2 %); super-zone R2 0.997, slope 1.053, RMSE% 7.4; within ±5 %: 14 of 36; after one factor (0.951): within ±5 %: 29, RMSE% 3.6
TAZ level: R2 0.493, slope 1.027, RMSE% 91
           sector  zonal residents  THS residents  diff %  zonal share  THS share  super-zones with > 1,000 zonal residents  of them within ±5 % after one factor
Jewish non-Haredi        1388901.0    1378409.081  -0.755        0.547      0.516                                        33                                     9
           Haredi          86225.0     108502.042  25.836        0.034      0.041                                        15                                     1
             Arab        1063969.0    1183807.900  11.263        0.419      0.443         

In [6]:
add_summary('T2', '3.3', 'residents by super-zone, THS-expanded vs zonal population 2020', f'{len(t2)} super-zones', 'each within ±5 %',
            f'total {fs["total_diff_pct"]:+.1f} %; {int(t2["within ±5 %"].sum())} of {len(t2)} super-zones within ±5 % ({int(t2["within ±5 % after one factor"].sum())} after one overall factor); R2 {fs["R2"]:.3f}, slope {fs["slope_origin"]:.2f}, RMSE% {fs["RMSE_pct"]:.0f}',
            vm.verdict(bool(t2['within ±5 %'].all()), explained=True), 'The survey year (2017/18) is not the zonal base year (2020), and the survey sample (5,098 households) is thin in small super-zones; the pattern after one factor shows the spatial agreement.', 'THS 2017/18 (new_wf) vs Zonal_2020.csv')
for _, r in t2s.iterrows():
    add_summary('T2', '3.3', f'residents of the {r["sector"]} sector, THS-expanded vs zonal', 'study area', '±5 %', f'{r["diff %"]:+.1f} % (THS share {r["THS share"]:.1%} vs zonal {r["zonal share"]:.1%})',
                vm.verdict(bool(abs(r['diff %']) <= 5), explained=True), 'Sector shares of the zonal file are population shares of the TAZ; the survey sector is the household type.', 'THS 2017/18 vs Zonal_2020.csv')
add_summary('T2', '3.3', 'average household size by super-zone', f'{len(t2h)} super-zones', 'each within ±5 %', f'study-wide zonal {zon_pop.sum() / zon_hh.sum():.2f} vs THS {tot_size:.2f}; {int(t2h["size within ±5 %"].sum())} of {len(t2h)} super-zones within ±5 %',
            vm.verdict(bool(t2h['size within ±5 %'].all()), explained=True), 'A systematic study-wide gap (survey households are larger by about 12 %, with 4 % fewer households and 5 % more residents than the zonal file), not sampling noise. Not resolved: the households definition of the zonal file (private households only?) and the survey household-size field have no dictionary here.', 'THS 2017/18 vs Zonal_2020.csv')
add_summary('T2', '3.3', 'household car availability, employed residents by residence / workplace, work location', 'super-zone', '±5 % / ±15 % against CBS', 'not run: the CBS social survey and labour-force tables are not in the repository', 'not run', 'Standing data request 6; the THS car-availability distribution is given as a descriptive table.', '')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), facecolor='white')
scatter_origin(axes[0], t2['zonal population 2020'], t2['THS-expanded residents'], 'Residents by super-zone', 'zonal population 2020 (observed)', 'THS-expanded residents (model)', color=BLUE)
scatter_origin(axes[1], zon.loc[TAZ, 'POPULATION'], tazp, 'Residents by TAZ', 'zonal population 2020 (observed)', 'THS-expanded residents (model)', color=AQUA)
save_fig(fig, 'mot_T2_residents.png')

## 3. T3 - trips per resident (guideline 3.4)

**Test.** Trip rates by segment (±15 %) from the expanded survey against the matrices. The rate here is *origin trips of the time window* (all motorised modes of the chain: car + bus + taxi-type + rail) per resident of the origin zone,
so it is a window rate, not a daily rate. Two comparisons:
* **2018 survey layer vs 2022 final layer** by super-zone: does the 2018 -> 2022 step (growth factors, RavKav anchoring) keep the trip rate within ±15 %? Resident bases: THS-expanded residents (T2) for 2018, zonal population interpolated to 2022 (`pop20 * (pop25/pop20)^(2/5)`, the chain's own growth rule) for 2022.
* **The same by sector and by locality type** (TAZ proxy, see section 0).
The bus anchoring to RavKav is meant to change the transit part of the rate (the survey under-covers some bus trips), so the transit part is shown separately from the car part.

In [7]:
two, thr = 'Output/ths2017/two_mode' + SUF, 'Output/ths2017/three_mode_2022' + SUF
def sq(p): m = pd.read_csv(p, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int); return m.reindex(index=TAZ, columns=TAZ).fillna(0)
need(f'{two}/car_taz.csv', f'{two}/transit_survey_taz.csv', f'{thr}/car_2022_taz.csv', f'{thr}/bus_2022_taz.csv', f'{thr}/taxi_2022_taz.csv', f'{thr}/rail_2022_taz.csv')
car18, tr18 = sq(f'{two}/car_taz.csv'), sq(f'{two}/transit_survey_taz.csv')
car22, bus22, taxi22, rail22 = (sq(f'{thr}/{n}_2022_taz.csv') for n in ['car', 'bus', 'taxi', 'rail'])
o18 = pd.DataFrame({'car': car18.sum(axis=1), 'transit': tr18.sum(axis=1)}); o22 = pd.DataFrame({'car': car22.sum(axis=1), 'transit': (bus22 + taxi22 + rail22).sum(axis=1)})
pop20, pop25 = zon.loc[TAZ, 'POPULATION'], zon25.reindex(TAZ)['POPULATION'].fillna(zon.loc[TAZ, 'POPULATION'])
pop22 = pop20 * np.divide(pop25, pop20, out=np.ones(len(TAZ)), where=pop20.values > 0) ** (2 / 5)
res18 = P.groupby('home_taz')['new_wf'].sum().reindex(TAZ).fillna(0)
def rate_table(key, order):
    g = pd.Series(key, index=TAZ); rows = []
    for k in order:
        t = g.index[g == k]; r18, r22 = res18[t].sum(), pop22[t].sum()
        if r18 <= 0 or r22 <= 0: continue
        row = {'segment': k, 'THS residents': r18, 'zonal residents 2022': r22}
        for m in ['car', 'transit']: row[f'{m} 2018'] = o18.loc[t, m].sum() / r18; row[f'{m} 2022'] = o22.loc[t, m].sum() / r22
        row['all 2018'] = row['car 2018'] + row['transit 2018']; row['all 2022'] = row['car 2022'] + row['transit 2022']; row['all diff %'] = 100 * (row['all 2022'] / row['all 2018'] - 1) if row['all 2018'] > 0 else np.nan
        row['car diff %'] = 100 * (row['car 2022'] / row['car 2018'] - 1) if row['car 2018'] > 0 else np.nan; row['transit diff %'] = 100 * (row['transit 2022'] / row['transit 2018'] - 1) if row['transit 2018'] > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)
t3sz = rate_table(taz_sz.values, SZL); t3sz['within ±15 %'] = t3sz['all diff %'].abs() <= 15; t3sz.to_csv(vfile('T3_rates_by_superzone'), index=False, float_format='%.4f')
t3sec = rate_table(taz_sector.values, SECTORS); t3sec['within ±15 %'] = t3sec['all diff %'].abs() <= 15; t3sec.to_csv(vfile('T3_rates_by_sector'), index=False, float_format='%.4f')
t3loc = rate_table(taz_loc.values, ['urban', 'suburban', 'rural']); t3loc['within ±15 %'] = t3loc['all diff %'].abs() <= 15; t3loc.to_csv(vfile('T3_rates_by_locality'), index=False, float_format='%.4f')
tot18, tot22 = (o18.sum().sum() / res18.sum()), (o22.sum().sum() / pop22.sum())
print(f'{PLABEL}: study-area window rate 2018 {tot18:.4f}, 2022 {tot22:.4f} ({100 * (tot22 / tot18 - 1):+.1f} %); super-zones within ±15 %: {int(t3sz["within ±15 %"].sum())} of {len(t3sz)}')
print(t3sec.round(3).to_string(index=False)); print(t3loc.round(3).to_string(index=False))
print('largest differences:'); print(t3sz.reindex(t3sz['all diff %'].abs().sort_values(ascending=False).index).head(8)[['segment', 'THS residents', 'all 2018', 'all 2022', 'car diff %', 'transit diff %', 'all diff %']].round(3).to_string(index=False))

midday 10:00-14:00: study-area window rate 2018 0.4615, 2022 0.4957 (+7.4 %); super-zones within ±15 %: 33 of 35
          segment  THS residents  zonal residents 2022  car 2018  car 2022  transit 2018  transit 2022  all 2018  all 2022  all diff %  car diff %  transit diff %  within ±15 %
Jewish non-Haredi    1469322.577           1472763.327     0.503     0.527         0.078         0.078     0.581     0.605       4.191       4.726           0.725          True
           Haredi      49941.826             47515.150     0.389     0.448         0.126         0.128     0.515     0.576      11.773      15.098           1.489          True
             Arab    1151454.619           1091571.321     0.287     0.324         0.020         0.021     0.307     0.345      12.267      12.768           4.961          True
 segment  THS residents  zonal residents 2022  car 2018  car 2022  transit 2018  transit 2022  all 2018  all 2022  all diff %  car diff %  transit diff %  within ±15 %
   urban   

In [8]:
big = t3sz[t3sz['THS residents'] >= 20000]
add_summary('T3', '3.4', f'window trip rate per resident, 2022 layers vs 2018 survey layers, by super-zone ({PLABEL})', f'{len(t3sz)} super-zones', 'each within ±15 %',
            f'study-area rate {tot18:.3f} -> {tot22:.3f} ({100 * (tot22 / tot18 - 1):+.1f} %); {int(t3sz["within ±15 %"].sum())} of {len(t3sz)} super-zones within ±15 % ({int(big["within ±15 %"].sum())} of {len(big)} with at least 20,000 residents)',
            vm.verdict(bool(t3sz['within ±15 %'].all()), explained=True), 'Outside ±15 %: ' + ', '.join(f'super-zone {int(r.segment)} ({r["all diff %"]:+.0f} %)' for _, r in t3sz[~t3sz['within ±15 %']].iterrows()) + '. The 2022 rate uses zonal residents (2020 base grown by the chain rule) against THS-expanded residents in 2018, so the survey-versus-zonal gap of T2 enters; bus anchoring to RavKav changes the transit part by design.', 'matrices 2018 / 2022; T2 residents')
for _, r in t3sec.iterrows():
    add_summary('T3', '3.4', f'window trip rate per resident, {r["segment"]} TAZs (dominant-sector proxy), 2022 vs 2018', 'sector', '±15 %', f'{r["all 2018"]:.3f} -> {r["all 2022"]:.3f} ({r["all diff %"]:+.1f} %); car {r["car diff %"]:+.1f} %, transit {r["transit diff %"]:+.1f} %',
                vm.verdict(bool(abs(r['all diff %']) <= 15), explained=True), 'Sector is a TAZ-level proxy (matrices carry no sector).', 'matrices; Zonal_2020.csv')
for _, r in t3loc.iterrows():
    add_summary('T3', '3.4', f'window trip rate per resident, {r["segment"]} TAZs, 2022 vs 2018', 'locality type', '±15 %', f'{r["all 2018"]:.3f} -> {r["all 2022"]:.3f} ({r["all diff %"]:+.1f} %)',
                vm.verdict(bool(abs(r['all diff %']) <= 15), explained=True), 'Locality type from SETTL_SIZE (assumed reading).', 'matrices; Zonal_2020.csv')
add_summary('T3', '3.4', 'trip rates by household size, car availability, employment status, and against CBS / social survey', 'segment', '±15 %', 'not run: the matrices carry no household segmentation and the CBS tables are not in the repository', 'not run', 'The model is a zone-to-zone layer set, not a household-segmented generation model; the survey rates by person type could be given as descriptive tables on request.', '')
fig, ax = plt.subplots(figsize=(6.2, 4.8), facecolor='white')
scatter_origin(ax, t3sz['all 2018'], t3sz['all 2022'], f'Window trips per resident by super-zone, {PLABEL}', '2018 survey layers', '2022 final layers', color=BLUE); save_fig(fig, 'mot_T3_rates.png')

## 4. T6 - intra-zonal trips (guideline 3.5)

**Test.** Share of trips that start and end in the same TAZ below 5 % (user's reading, decision of 4 October 2026), by layer, super-zone, locality type and sector proxy. The survey is a sample with a TAZ-allocation step (each 1250-zone's trips are split among its TAZs
by population on the origin side and employment on the destination side), so the diagonal is partly created by that split; a second figure gives the share inside the **1250-zone**, the unit the survey actually observes.

In [9]:
LAY = {'car 2022': car22, 'bus 2022': bus22, 'taxi 2022': taxi22, 'rail 2022': rail22, 'car 2018 survey': car18, 'transit 2018 survey': tr18}
LAY['all modes 2022'] = car22 + bus22 + taxi22 + rail22; LAY['car + transit 2022'] = LAY['all modes 2022']
z_of = pd.Series(taz_1250.values, index=TAZ)
same1250 = (z_of.values[:, None] == z_of.values[None, :])
rows = []
for n, M in LAY.items():
    A = M.values; tot = A.sum(); rows.append({'layer': n, 'trips': tot, 'intra-TAZ trips': np.trace(A), 'intra-TAZ share': np.trace(A) / tot if tot else np.nan, 'intra-1250-zone share': A[same1250].sum() / tot if tot else np.nan,
                                              'intra-super-zone share': A[(taz_sz.values[:, None] == taz_sz.values[None, :])].sum() / tot if tot else np.nan})
t6 = pd.DataFrame(rows).drop_duplicates('layer'); t6.to_csv(vfile('T6_intra_by_layer'), index=False, float_format='%.4f'); print(t6.round(4).to_string(index=False))
AL = LAY['all modes 2022']; A18 = car18 + tr18
def intra_by(key, order, M):
    g = pd.Series(key, index=TAZ); out = []
    for k in order:
        t = np.where(g.values == k)[0]; row_tot = M.values[t, :].sum(); out.append({'segment': k, 'origin trips': row_tot, 'intra-TAZ share': M.values[t, t].sum() / row_tot if row_tot else np.nan})
    return pd.DataFrame(out)
t6sz = intra_by(taz_sz.values, SZL, AL); t6sz['intra-TAZ share 2018 survey'] = intra_by(taz_sz.values, SZL, A18)['intra-TAZ share']; t6sz['within 5 %'] = t6sz['intra-TAZ share'] < 0.05
t6loc = intra_by(taz_loc.values, ['urban', 'suburban', 'rural'], AL); t6sec = intra_by(taz_sector.values, SECTORS, AL)
for n, d in [('superzone', t6sz), ('locality', t6loc), ('sector', t6sec)]: d.to_csv(vfile(f'T6_intra_by_{n}'), index=False, float_format='%.4f')
print(t6loc.round(4).to_string(index=False)); print(t6sec.round(4).to_string(index=False)); print(f'super-zones with an intra-TAZ share below 5 %: {int(t6sz["within 5 %"].sum())} of {len(t6sz)}')
tot_share = np.trace(AL.values) / AL.values.sum(); share_1250 = AL.values[same1250].sum() / AL.values.sum()
print(f'{PLABEL}: all modes 2022 intra-TAZ {tot_share:.1%}, intra-1250-zone {share_1250:.1%}; 2018 survey layers {np.trace(A18.values) / A18.values.sum():.1%}')

              layer        trips  intra-TAZ trips  intra-TAZ share  intra-1250-zone share  intra-super-zone share
           car 2022 1150867.9131      201965.8833           0.1755                 0.3289                  0.6711
           bus 2022  129528.0497        7323.4184           0.0565                 0.1091                  0.5556
          taxi 2022   12537.9330         427.7541           0.0341                 0.0787                  0.6096
          rail 2022    1701.8103           0.0000           0.0000                 0.0000                  0.0000
    car 2018 survey 1089436.2365      190459.5108           0.1748                 0.3286                  0.6718
transit 2018 survey  143001.8150        6956.7335           0.0486                 0.1103                  0.5597
     all modes 2022 1294635.7061      209717.0558           0.1620                 0.3040                  0.6580
 car + transit 2022 1294635.7061      209717.0558           0.1620                 0.304

 segment  origin trips  intra-TAZ share
   urban   754103.3917           0.1232
suburban   409099.4410           0.2545
   rural   131432.8734           0.0969
          segment  origin trips  intra-TAZ share
Jewish non-Haredi   890973.3454           0.1397
           Haredi    27368.2005           0.2616
             Arab   376294.1603           0.2075
super-zones with an intra-TAZ share below 5 %: 6 of 36
midday 10:00-14:00: all modes 2022 intra-TAZ 16.2%, intra-1250-zone 30.4%; 2018 survey layers 16.0%


In [10]:
add_summary('T6', '3.5', f'intra-TAZ share of trips, all modes 2022 ({PLABEL})', 'study area', '< 5 %', f'{tot_share:.1%} of trips start and end in the same TAZ ({share_1250:.1%} inside the same 1250-zone, {np.trace(A18.values) / A18.values.sum():.1%} in the 2018 survey layers)',
            vm.verdict(bool(tot_share < 0.05), explained=True), 'The survey itself puts about 38 % of trips inside one 1250-zone; with the TAZ split of each 1250-zone by population and employment the diagonal is 20 % at TAZ level, in the 2018 survey layers as in the 2022 layers (growth does not change it). The share is a property of the survey trip length and of the zone size, not a result of the 2022 steps; no parameter is changed. The by-super-zone table shows where it is highest.', 'matrices 2018 / 2022')
add_summary('T6', '3.5', 'intra-TAZ share by super-zone', f'{len(t6sz)} super-zones', 'each < 5 %', f'{int(t6sz["within 5 %"].sum())} of {len(t6sz)} below 5 %; highest {t6sz.loc[t6sz["intra-TAZ share"].idxmax(), "segment"]} ({t6sz["intra-TAZ share"].max():.0%})',
            vm.verdict(bool(t6sz['within 5 %'].all()), explained=True), 'See study-area row.', 'matrices 2022')
for _, r in t6loc.iterrows():
    add_summary('T6', '3.5', f'intra-TAZ share, {r["segment"]} TAZs', 'locality type', '< 5 %', f'{r["intra-TAZ share"]:.1%}', vm.verdict(bool(r['intra-TAZ share'] < 0.05), explained=True), 'Locality type from SETTL_SIZE (assumed reading).', 'matrices 2022')
for _, r in t6sec.iterrows():
    add_summary('T6', '3.5', f'intra-TAZ share, {r["segment"]} TAZs (dominant-sector proxy)', 'sector', '< 5 %', f'{r["intra-TAZ share"]:.1%}', vm.verdict(bool(r['intra-TAZ share'] < 0.05), explained=True), 'Sector is a TAZ-level proxy.', 'matrices 2022')
fig, ax = plt.subplots(figsize=(9, 4.2), facecolor='white')
t_ = t6sz.sort_values('intra-TAZ share', ascending=False); ax.bar(range(len(t_)), t_['intra-TAZ share'] * 100, color=BLUE); ax.axhline(5, color=ORANGE, lw=1.2); ax.set_xticks(range(len(t_))); ax.set_xticklabels(t_['segment'].astype(int), fontsize=7)
style_ax(ax, f'Intra-TAZ share of origin trips by super-zone, all modes 2022, {PLABEL}', 'super-zone', 'percent (orange: 5 % limit)'); fig.tight_layout(); save_fig(fig, 'mot_T6_intra.png')

SUMMARY[:] = [r for r in SUMMARY if PERIOD == 'AM' or r['test'] in ('T3', 'T6')]      # T1 and T2 do not depend on the window: reported once, with the AM run

## Summary of this notebook against the guideline

In [11]:
SUM = pd.DataFrame(SUMMARY); SUM.to_csv(f'{OUT}/summary_stage1d{PSUF}.csv', index=False)
SUM[['test', 'guideline', 'item', 'level', 'criterion', 'result', 'status']]

,test,guideline,item,level,criterion,result,status
0,T3,3.4,"window trip rate per resident, 2022 layers vs ...",35 super-zones,each within ±15 %,study-area rate 0.461 -> 0.496 (+7.4 %); 33 of...,miss (explained)
1,T3,3.4,"window trip rate per resident, Jewish non-Hare...",sector,±15 %,"0.581 -> 0.605 (+4.2 %); car +4.7 %, transit +...",pass
2,T3,3.4,"window trip rate per resident, Haredi TAZs (do...",sector,±15 %,"0.515 -> 0.576 (+11.8 %); car +15.1 %, transit...",pass
3,T3,3.4,"window trip rate per resident, Arab TAZs (domi...",sector,±15 %,"0.307 -> 0.345 (+12.3 %); car +12.8 %, transit...",pass
4,T3,3.4,"window trip rate per resident, urban TAZs, 202...",locality type,±15 %,0.476 -> 0.515 (+8.2 %),pass
5,T3,3.4,"window trip rate per resident, suburban TAZs, ...",locality type,±15 %,0.356 -> 0.383 (+7.8 %),pass
6,T3,3.4,"window trip rate per resident, rural TAZs, 202...",locality type,±15 %,1.620 -> 1.605 (-1.0 %),pass
7,T3,3.4,"trip rates by household size, car availability...",segment,±15 %,not run: the matrices carry no household segme...,not run
8,T6,3.5,"intra-TAZ share of trips, all modes 2022 (midd...",study area,< 5 %,16.2% of trips start and end in the same TAZ (...,miss (explained)
9,T6,3.5,intra-TAZ share by super-zone,36 super-zones,each < 5 %,6 of 36 below 5 %; highest 31 (42%),miss (explained)
